In [1]:
"""
Этап 4. XYZ-анализ: стабильность спроса по месяцам.

Ключевой момент: строим ПОЛНЫЙ календарь 24 месяца
(2009-12 — 2011-11). Отсутствующие месяцы = 0, а не пропуск.
"""
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import psycopg2

ROOT_DIR = Path.cwd().parent
sys.path.append(str(ROOT_DIR))

from scripts.db_config import read_sql, execute_sql, _get_conn_params

pd.set_option("display.max_rows", 100)
pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (12, 5)

REPORTS_DIR = ROOT_DIR / "reports"
REPORTS_DIR.mkdir(exist_ok=True)

print("✅ Окружение готово")

✅ Окружение готово


In [2]:
"""
Фаза 4. Диагностика отмен.

Проверяем:
- Сколько отмен всего?
- Есть ли крупные покупки, которые затем отменены?
- Как это влияет на анализ?
"""
import sys
from pathlib import Path

import pandas as pd
import numpy as np

ROOT_DIR = Path.cwd().parent
sys.path.append(str(ROOT_DIR))

from scripts.db_config import read_sql

pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

# 1. Общая статистика отмен
cancel_summary = read_sql("""
    SELECT
        COUNT(*) AS total_cancellations,
        COUNT(DISTINCT invoice_no) AS cancelled_invoices,
        ROUND(SUM(quantity)::numeric, 0) AS total_qty_cancelled,
        ROUND(SUM(quantity * unit_price)::numeric, 2) AS total_revenue_cancelled
    FROM raw_transactions
    WHERE invoice_no LIKE 'C%';
""")
print("Общая статистика отмен:")
display(cancel_summary.T)

# 2. Крупные отмены (по количеству)
large_cancels = read_sql("""
    SELECT
        invoice_no,
        stock_code,
        description,
        quantity,
        unit_price,
        invoice_date,
        ROUND((quantity * unit_price)::numeric, 2) AS revenue
    FROM raw_transactions
    WHERE invoice_no LIKE 'C%'
      AND ABS(quantity) >= 1000
    ORDER BY ABS(quantity) DESC
    LIMIT 30;
""")
print("\nТоп-30 крупных отмен (по количеству):")
display(large_cancels)

Общая статистика отмен:


,0
total_cancellations,19494
cancelled_invoices,8292
total_qty_cancelled,-490992
total_revenue_cancelled,-1526667.86



Топ-30 крупных отмен (по количеству):


,invoice_no,stock_code,description,quantity,unit_price,invoice_date,revenue
0,C581484,23843,"PAPER CRAFT , LITTLE BIRDIE",-80995,2.08,2011-12-09 09:27:00,-168469.60
1,C541433,23166,MEDIUM CERAMIC TOP STORAGE JAR,-74215,1.04,2011-01-18 10:17:00,-77183.60
2,C536757,84347,ROTATING SILVER ANGELS T-LIGHT HLDR,-9360,0.03,2010-12-02 14:23:00,-280.80
3,C536757,84347,ROTATING SILVER ANGELS T-LIGHT HLDR,-9360,0.03,2010-12-02 14:23:00,-280.80
4,C524235,21088,SET/6 FRUIT SALAD PAPER CUPS,-7128,0.08,2010-09-28 11:02:00,-570.24
5,C524235,21096,SET/6 FRUIT SALAD PAPER PLATES,-7008,0.13,2010-09-28 11:02:00,-911.04
6,C524235,16047,POP ART PEN CASE & PENS,-5184,0.08,2010-09-28 11:02:00,-414.72
7,C524235,37340,MULTICOLOUR SPRING FLOWER MUG,-4992,0.1,2010-09-28 11:02:00,-499.20
8,C524235,85110,BLACK SILVER FLOWER T-LIGHT HOLDER,-4752,0.07,2010-09-28 11:02:00,-332.64
9,C524235,16046,TEATIME PEN CASE & PENS,-4608,0.08,2010-09-28 11:02:00,-368.64


In [5]:
# 3. Поиск крупных пар "покупка → отмена"
# Логика: одна и та же stock_code, |quantity| совпадает, разные invoice_no,
# и оба invoice_no содержат одинаковую часть (например, C12345 → 12345)

matched_cancellations = read_sql("""
-- Находим все пары "покупка ↔ отмена"
    WITH positives AS (
        SELECT
            invoice_no,
            stock_code,
            quantity,
            unit_price,
            customer_id,
            invoice_date
        FROM raw_transactions
        WHERE invoice_no NOT LIKE 'C%'
        AND quantity >= 100        -- порог
        AND customer_id IS NOT NULL
    ),
    negatives AS (
        SELECT
            invoice_no AS cancel_invoice,
            stock_code,
            ABS(quantity) AS quantity,
            customer_id,
            invoice_date AS cancel_date
        FROM raw_transactions
        WHERE invoice_no LIKE 'C%'
        AND ABS(quantity) >= 100
        AND customer_id IS NOT NULL
    )
    SELECT
        p.invoice_no    AS purchase_invoice,
        n.cancel_invoice,
        p.stock_code,
        p.quantity,
        p.customer_id,
        p.invoice_date  AS purchase_date,
        n.cancel_date,
        EXTRACT(EPOCH FROM (n.cancel_date - p.invoice_date)) / 60 AS minutes_diff
    FROM positives p
    INNER JOIN negatives n
        ON p.stock_code = n.stock_code
        AND p.quantity = n.quantity
        AND p.customer_id = n.customer_id
        AND n.cancel_date >= p.invoice_date
        AND n.cancel_date <= p.invoice_date + INTERVAL '24 hours'  -- в пределах суток
    WHERE EXTRACT(EPOCH FROM (n.cancel_date - p.invoice_date)) / 60 <= 60  -- до 60 минут
    ORDER BY p.quantity DESC;
""")
print("\nКрупные пары 'покупка → отмена':")
display(matched_cancellations)


Крупные пары 'покупка → отмена':


,purchase_invoice,cancel_invoice,stock_code,quantity,customer_id,purchase_date,cancel_date,minutes_diff
0,581483,C581484,23843,80995,16446,2011-12-09 09:15:00,2011-12-09 09:27:00,12.0000000000000000
1,541431,C541433,23166,74215,12346,2011-01-18 10:01:00,2011-01-18 10:17:00,16.0000000000000000
2,508333,C508334,47587A,1200,13694,2010-05-14 12:00:00,2010-05-14 12:04:00,4.0000000000000000
3,529350,C529352,71477,1152,17450,2010-10-28 09:29:00,2010-10-28 09:32:00,3.0000000000000000
4,511164,C511172,20726,830,14028,2010-06-07 12:02:00,2010-06-07 12:05:00,3.0000000000000000
...,...,...,...,...,...,...,...,...
129,575220,C575224,23581,100,17511,2011-11-09 11:19:00,2011-11-09 11:41:00,22.0000000000000000
130,575220,C575224,85099C,100,17511,2011-11-09 11:19:00,2011-11-09 11:41:00,22.0000000000000000
131,575220,C575224,23199,100,17511,2011-11-09 11:19:00,2011-11-09 11:41:00,22.0000000000000000
132,575220,C575224,23582,100,17511,2011-11-09 11:19:00,2011-11-09 11:41:00,22.0000000000000000


In [4]:
# 4. Конкретная проверка SKU 23166
check_23166 = read_sql("""
    SELECT
        invoice_no,
        stock_code,
        description,
        quantity,
        unit_price,
        invoice_date,
        ROUND((quantity * unit_price)::numeric, 2) AS revenue
    FROM raw_transactions
    WHERE stock_code = '23166'
    ORDER BY invoice_date;
""")
print(f"\nВсе операции по SKU 23166: {len(check_23166)} строк")
display(check_23166)


Все операции по SKU 23166: 260 строк


,invoice_no,stock_code,description,quantity,unit_price,invoice_date,revenue
0,541431,23166,MEDIUM CERAMIC TOP STORAGE JAR,74215,1.04,2011-01-18 10:01:00,77183.60
1,C541433,23166,MEDIUM CERAMIC TOP STORAGE JAR,-74215,1.04,2011-01-18 10:17:00,-77183.60
2,552882,23166,MEDIUM CERAMIC TOP STORAGE JAR,96,1.04,2011-05-12 10:10:00,99.84
3,552953,23166,MEDIUM CERAMIC TOP STORAGE JAR,4,1.25,2011-05-12 12:11:00,5.00
4,553005,23166,MEDIUM CERAMIC TOP STORAGE JAR,5,1.25,2011-05-12 16:29:00,6.25
...,...,...,...,...,...,...,...
255,581108,23166,MEDIUM CERAMIC TOP STORAGE JAR,2,1.25,2011-12-07 12:16:00,2.50
256,581219,23166,MEDIUM CERAMIC TOP STORAGE JAR,1,2.46,2011-12-08 09:28:00,2.46
257,581439,23166,MEDIUM CERAMIC TOP STORAGE JAR,2,2.46,2011-12-08 16:30:00,4.92
258,581476,23166,MEDIUM CERAMIC TOP STORAGE JAR,48,1.04,2011-12-09 08:48:00,49.92


In [6]:
# Сохраняем все пары в БД для дальнейшего использования
execute_sql("DROP TABLE IF EXISTS cancelled_pairs;")
execute_sql("""
    CREATE TABLE cancelled_pairs AS
    WITH positives AS (
        SELECT
            invoice_no,
            stock_code,
            quantity,
            customer_id,
            invoice_date
        FROM raw_transactions
        WHERE invoice_no NOT LIKE 'C%'
          AND quantity >= 100
          AND customer_id IS NOT NULL
    ),
    negatives AS (
        SELECT
            invoice_no AS cancel_invoice,
            stock_code,
            ABS(quantity) AS quantity,
            customer_id,
            invoice_date AS cancel_date
        FROM raw_transactions
        WHERE invoice_no LIKE 'C%'
          AND ABS(quantity) >= 100
          AND customer_id IS NOT NULL
    )
    SELECT DISTINCT
        p.invoice_no    AS purchase_invoice,
        n.cancel_invoice,
        p.stock_code,
        p.quantity,
        p.customer_id,
        p.invoice_date  AS purchase_date,
        n.cancel_date
    FROM positives p
    INNER JOIN negatives n
        ON p.stock_code = n.stock_code
        AND p.quantity = n.quantity
        AND p.customer_id = n.customer_id
        AND n.cancel_date >= p.invoice_date
        AND n.cancel_date <= p.invoice_date + INTERVAL '60 minutes'
""")

# Проверка
check = read_sql("""
    SELECT
        COUNT(*) AS pairs_count,
        COUNT(DISTINCT stock_code) AS unique_skus,
        SUM(quantity) AS total_qty
    FROM cancelled_pairs;
""")
print("Сводка cancelled_pairs:")
display(check.T)

Сводка cancelled_pairs:


,0
pairs_count,134
unique_skus,89
total_qty,182231
